# 01 — Data collection

**Research question:** Which factors are associated with public transport delays in Switzerland, and how do these effects differ between regions, transport modes and time periods?

This notebook documents group-owned collection from official web/API sources. Raw responses are preserved without overwriting. For the final study, collect repeated stationboard snapshots across at least 28 service days. Actual data v2 remains an optional validation or historical extension.

In [ ]:
import pandas as pd
from sptdelays.settings import PATHS, load_settings
from sptdelays.collect_stations import resolve_stations, load_station_panel
from sptdelays.collect_live import collect_live
from sptdelays.collect_actuals import discover_resources, collect_actuals

RUN_COLLECTION = False  # Set True only when intentionally collecting new API data.
settings = load_settings()
stations = load_station_panel()
stations[['station_id', 'station_name', 'region', 'station_type']].head()

## Resolve and validate the station panel
The `/locations` endpoint supplies stable identifiers and coordinates. Inspect every returned match manually before final collection.

In [ ]:
resolved = resolve_stations() if RUN_COLLECTION else load_station_panel()
resolved[['station_name', 'station_id', 'region', 'latitude', 'longitude']]

## Primary source: repeated stationboard API snapshots
Run this every 15–30 minutes across the approved collection window. Each run writes a timestamped raw JSON response, appends normalized records and updates the collection log.

In [ ]:
live = collect_live() if RUN_COLLECTION else pd.read_csv(PATHS.interim / 'live_observations.csv')
live[['observed_at', 'station_name', 'category_raw', 'scheduled_time', 'delay_minutes_signed']].head()

## Optional historical source: Actual data v2
The dataset page is scraped to discover official daily CSV links. Use selected dates only when the group and lecturer approve this extension because individual daily files are large. Keep a record of every selected date and URL.

In [ ]:
resources = discover_resources() if RUN_COLLECTION else {}
pd.DataFrame(sorted(resources.items()), columns=['service_date', 'resource_url']).tail()

In [ ]:
# Deliberately disabled: enter approved dates, then set RUN_FINAL_COLLECTION = True.
RUN_FINAL_COLLECTION = False
approved_dates = ['YYYY-MM-DD']
if RUN_FINAL_COLLECTION:
    collected_paths = collect_actuals(approved_dates)
    collected_paths

## Required evidence
Record collection dates, intervals, URLs, row counts, errors and access dates. Preserve `collection_log.csv`. Treat API prognosis and Actual data values as reported timing information rather than exact platform measurements.